# Stage 8 — CNN search and final seeds (PR C)

This notebook runs the official D30 CNN search from an exact clean commit: it builds the train and val-A log-Mel caches twice and compares their bytes, runs `src.tuning.cnn_search` once (4 search runs, then the selected configuration's final seeds 1 and 2) and exports the checkpoints and reports to Drive. Only train and val-A are read: val-B is not cached or scored before Stage 10 (D42), and the test split is never requested. Every hyperparameter comes from `configs/cnn.yaml`. Run it only after the implementation commit is reviewed and merged.

## 1. Parameters

Replace `COMMIT` with the full 40-character approved commit. `VERSION` names the checkpoint folder and the W&B runs. There are no other knobs: the grid, epochs, patience, batch size and seeds are read from `configs/cnn.yaml` by the search code.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'  # full 40-character SHA
VERSION = 'cnn-v1'
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'

## 2. Mount Drive and refuse existing outputs

Drive is mounted before anything else. A frozen checkpoint or evidence folder is never overwritten, so the notebook stops here, before any expensive work, if `checkpoints/cnn/<VERSION>/` or `processed/metadata/stage8/` already exists. The same check runs again right before export.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
CHECKPOINT_DEST = DRIVE / 'checkpoints' / 'cnn' / VERSION
EVIDENCE_DEST = DRIVE / 'processed' / 'metadata' / 'stage8'


def refuse_existing_outputs():
    for dest in (CHECKPOINT_DEST, EVIDENCE_DEST):
        if dest.exists():
            raise FileExistsError(f'{dest} already exists; Stage 8 outputs are never overwritten')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


refuse_existing_outputs()
assert (DRIVE / 'raw').is_dir(), f'{DRIVE / "raw"} not found; check DRIVE_ROOT'

## 3. Check out the exact commit

The repository is cloned into `/content` and the full commit is checked out detached. HEAD must equal `COMMIT` exactly and the worktree must be clean. Every later output (caches, pytest XML, W&B files, search output) is written outside the clone, so both CLIs, which refuse a dirty worktree, see a clean tree. `assert_clean` also refuses ignored files other than Python bytecode caches, because `.gitignore` hides `wandb/` from `git status`.

In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout


def assert_clean():
    head = git('rev-parse', 'HEAD').strip()
    assert head == COMMIT, (head, COMMIT)
    status = git('status', '--porcelain')
    assert not status, status
    ignored = [line[3:] for line in git('status', '--porcelain', '--ignored').splitlines() if line.startswith('!! ')]
    unexpected = [path for path in ignored if not path.rstrip('/').endswith('__pycache__')]
    assert not unexpected, unexpected


assert_clean()
os.chdir(REPO)
print('HEAD', COMMIT)

## 4. Environment: Colab's torch, then the pinned requirements

The runtime must be Python 3.13 (D15). Colab's preinstalled CUDA build of torch is used and must be 2.11.0 (`requirements-cnn.txt`, D42). Torch is never installed by this notebook: only `requirements.txt` and `requirements-cnn.txt` are installed, and neither lists torch. The versions are then read again in a fresh process, so a silent upgrade during the install is caught.

In [ ]:
import torch

assert torch.__version__.startswith('2.11.0'), torch.__version__
assert torch.cuda.is_available(), 'a GPU runtime is required'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-cnn.txt'],
               cwd=REPO, check=True)
PROBE = '''
import json, platform, torch
cuda = torch.cuda.is_available()
print(json.dumps({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': cuda,
    'cuda_version': torch.version.cuda,
    'cudnn_version': torch.backends.cudnn.version() if cuda else None,
    'gpu_name': torch.cuda.get_device_name(0) if cuda else None,
}))
'''
ENV = json.loads(subprocess.run([sys.executable, '-c', PROBE], check=True, capture_output=True, text=True).stdout)
assert ENV['torch'] == torch.__version__ and ENV['torch'].startswith('2.11.0'), (ENV['torch'], torch.__version__)
assert ENV['python'].startswith('3.13.'), f"Python 3.13 is required (D15), got {ENV['python']}"
assert ENV['cuda_available'], 'CUDA is not available after the install'
assert_clean()
for key, value in ENV.items():
    print(f'{key}: {value}')

## 5. Verify and extract the pinned FoR archive

As in notebook 03, the revision marker next to the Drive backup must equal the D21 FoR revision, read here from `configs/audit.yaml` (the D21 pin location). The archive is extracted with `filter='data'`. Its SHA-256 is computed and written to the export manifest as `for_archive_sha256_observed`: this is evidence only, never asserted or compared, and not a dataset pin (D21 unchanged).

In [ ]:
import tarfile
import yaml

EXPECTED_REVISION = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']['revision']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION_FILE = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / 'for_2sec_ff8c82c7.tar'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
FOR_ARCHIVE_SHA256_OBSERVED = sha256(LOCAL_ARCHIVE)  # recorded only (evidence, not a pin)
print('FoR archive extracted to', AUDIO_ROOT)
print('for_archive_sha256_observed:', FOR_ARCHIVE_SHA256_OBSERVED)

## 6. Run the test suite: no failure, error or skip

`REQUIRE_TORCH_TESTS=1` turns a missing torch into a failure. The counts come from the JUnit XML written to `/content` (outside the clone), not from the terminal text, and every failure, error and skip stops the notebook.

In [ ]:
import xml.etree.ElementTree as ET


def stream(command, **kwargs):
    # Like subprocess.run(check=True), but prints the child's output line by line as it runs.
    env = {**kwargs.pop('env', os.environ), 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                          **kwargs) as process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


JUNIT = Path('/content/pytest_stage8.xml')
if JUNIT.exists():
    JUNIT.unlink()
assert_clean()
stream([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', f'--junitxml={JUNIT}'],
       cwd=REPO, env={**os.environ, 'REQUIRE_TORCH_TESTS': '1'})
root = ET.parse(JUNIT).getroot()
suites = [root] if root.tag == 'testsuite' else root.findall('testsuite')
COUNTS = {key: sum(int(suite.get(key, 0)) for suite in suites) for key in ('tests', 'failures', 'errors', 'skipped')}
print(COUNTS)
assert COUNTS['tests'] > 0, COUNTS
assert COUNTS['failures'] == COUNTS['errors'] == COUNTS['skipped'] == 0, COUNTS
assert_clean()

## 7. Build the train and val-A log-Mel caches twice and compare the bytes

Each split is cached into `/content/cache_a` and again into `/content/cache_b`; every sample hash is checked before decoding. Both builds must hold the same files with identical SHA-256s, and every cache must record the F3 profile config SHA-256 and fingerprint stored in the repo (Stage 6 evidence, D39). val-B is not built (D42: it is not scored until Stage 10).

In [ ]:
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
F3 = json.loads((REPO / 'docs/audit_evidence/stage6_preprocessing/profile_report.json').read_text(encoding='utf-8'))['profile']
assert sha256(PROFILE) == F3['config_sha256'], 'configs/preprocessing.yaml differs from F3'
CACHE_SPLITS = ('train', 'val-A')
CACHE_A, CACHE_B = Path('/content/cache_a'), Path('/content/cache_b')
for cache in (CACHE_A, CACHE_B):
    if cache.exists():
        shutil.rmtree(cache)
    cache.mkdir()
    for split in CACHE_SPLITS:
        assert_clean()
        subprocess.run([
            sys.executable, '-m', 'src.features.logmel_cache',
            '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE),
            '--split', split, '--out', str(cache / split),
        ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)


def file_hashes(folder):
    return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(folder.rglob('*')) if p.is_file()}


CACHE_HASHES = file_hashes(CACHE_A)
assert sorted(CACHE_HASHES) == sorted(f'{s}/{s}.{ext}' for s in CACHE_SPLITS for ext in ('npz', 'json')), sorted(CACHE_HASHES)
assert file_hashes(CACHE_B) == CACHE_HASHES, 'the two cache builds differ'
for split in CACHE_SPLITS:
    meta = json.loads((CACHE_A / split / f'{split}.json').read_text(encoding='utf-8'))
    assert meta['profile_config_sha256'] == F3['config_sha256'], (split, meta['profile_config_sha256'])
    assert meta['profile_fingerprint'] == F3['fingerprint'], (split, meta['profile_fingerprint'])
    assert meta['code_commit'] == COMMIT, (split, meta['code_commit'])
assert_clean()
CACHE_HASHES

## 8. W&B credentials and runtime folders

The API key is loaded last, just before the search, so that pytest and the cache builds never see it. It is read from the Colab secret `WANDB_API_KEY` (Secrets panel) into the environment, where wandb reads it in the search process (D40). It is never printed, and `wandb.login` is not called here. W&B's run, cache and config folders are placed under `/content/wandb_runtime`, outside the clone.

In [ ]:
from google.colab import userdata

_key = userdata.get('WANDB_API_KEY')
assert _key and _key.strip(), 'the W&B Colab secret is missing or empty'
os.environ['WANDB_API_KEY'] = _key
del _key
os.environ['WANDB_DIR'] = '/content/wandb_runtime/run'
os.environ['WANDB_CACHE_DIR'] = '/content/wandb_runtime/cache'
os.environ['WANDB_CONFIG_DIR'] = '/content/wandb_runtime/config'
for name in ('WANDB_DIR', 'WANDB_CACHE_DIR', 'WANDB_CONFIG_DIR'):
    folder = Path(os.environ[name])
    assert not folder.resolve().is_relative_to(REPO.resolve()), folder
    folder.mkdir(parents=True, exist_ok=True)
print('W&B key loaded from Colab secrets (not shown); runtime folders under /content/wandb_runtime')

## 9. Run the CNN search once

One call to `src.tuning.cnn_search` reads `cache_a`, trains the 4 search configurations with seed 0, selects one on val-A EER (D30 tie rule) and trains its seeds 1 and 2, logging every run to W&B online (D40). It runs exactly once: there is no retry, and nothing here reads or reacts to EER values. The output folder `/content/stage8_out` is outside the clone and must not exist yet.

In [ ]:
OUT = Path('/content/stage8_out')
if OUT.exists():
    raise FileExistsError(f'{OUT} already exists; the search runs once per runtime')
assert_clean()
stream([
    sys.executable, '-m', 'src.tuning.cnn_search',
    '--cache-dir', str(CACHE_A), '--profile', str(PROFILE), '--config', str(REPO / 'configs/cnn.yaml'),
    '--manifest', str(MANIFEST), '--out', str(OUT), '--version', VERSION,
    '--wandb-mode', 'online', '--device', 'cuda',
], cwd=REPO)

## 10. Post-run checks

The worktree must still be clean at `COMMIT`, which shows that nothing (wandb included) wrote into the clone. The selection report and the per-run peak GPU memory are printed for review; they are not used for any decision here.

In [ ]:
assert_clean()
REPORTS = ['search_report.json', 'selection_report.json', 'run_report.json']
RUN_REPORT = json.loads((OUT / 'run_report.json').read_text(encoding='utf-8'))
assert RUN_REPORT['code_commit'] == COMMIT, RUN_REPORT['code_commit']
assert RUN_REPORT['profile_fingerprint'] == F3['fingerprint'], RUN_REPORT['profile_fingerprint']
assert RUN_REPORT['profile_config_sha256'] == F3['config_sha256'], RUN_REPORT['profile_config_sha256']
print((OUT / 'selection_report.json').read_text(encoding='utf-8'))
for run, peak in sorted(RUN_REPORT['peak_gpu_memory_bytes'].items()):
    print(f'{run}: peak GPU memory {peak} bytes ({peak / 2**30:.2f} GiB)')

## 11. Export to Drive and write the export manifest

The checkpoints go to `checkpoints/cnn/<VERSION>/` and the three reports to `processed/metadata/stage8/`. Both folders are re-checked and created with `exist_ok=False`, so nothing is overwritten. Every Drive copy is re-hashed against its `/content` source, and `stage8_export_manifest.json` (paths relative to `DRIVE_ROOT`, sizes, SHA-256s, commit, version, environment, UTC time and the observed FoR archive SHA-256, which is evidence only) is written next to the reports. It also records the D21 FoR revision, the pytest counts, the SHA-256 of every cache file (both builds were identical) and the F3 config SHA-256 and fingerprint the caches were checked against. Its printout is the PR D evidence.

Two hash checks run. The first, right after copying, checks the copy. Drive is then flushed, unmounted and remounted, and every file is hashed again against the manifest: this second check covers what Drive actually stored after the flush, and also fails on a missing or extra file. These are candidate records until human review; checkpoints are never committed to Git.

In [ ]:
from datetime import datetime, timezone

refuse_existing_outputs()
SEED_DIRS = sorted((OUT / 'checkpoints').glob('seed_*'))
assert [d.name for d in SEED_DIRS] == ['seed_0', 'seed_1', 'seed_2'], [d.name for d in SEED_DIRS]
CHECKPOINT_DEST.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DEST.mkdir(exist_ok=False)
EVIDENCE_DEST.parent.mkdir(parents=True, exist_ok=True)
EVIDENCE_DEST.mkdir(exist_ok=False)
copies = []
for seed_dir in SEED_DIRS:
    shutil.copytree(seed_dir, CHECKPOINT_DEST / seed_dir.name)
    copies += [(p, CHECKPOINT_DEST / seed_dir.name / p.relative_to(seed_dir))
               for p in sorted(seed_dir.rglob('*')) if p.is_file()]
for name in REPORTS:
    shutil.copy2(OUT / name, EVIDENCE_DEST / name)
    copies.append((OUT / name, EVIDENCE_DEST / name))
files = []
for source, dest in copies:
    digest = sha256(source)
    assert sha256(dest) == digest, f'{dest} differs from {source}'
    files.append({'path': dest.relative_to(DRIVE).as_posix(), 'size': dest.stat().st_size, 'sha256': digest})
EXPORT_MANIFEST = {
    'commit': COMMIT,
    'version': VERSION,
    'python': ENV['python'],
    'torch': ENV['torch'],
    'cuda_version': ENV['cuda_version'],
    'cudnn_version': ENV['cudnn_version'],
    'gpu_name': ENV['gpu_name'],
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'for_revision': EXPECTED_REVISION,
    'for_archive_sha256_observed': FOR_ARCHIVE_SHA256_OBSERVED,
    'pytest_counts': COUNTS,
    'cache_sha256': CACHE_HASHES,
    'f3': {'config_sha256': F3['config_sha256'], 'fingerprint': F3['fingerprint']},
    'files': files,
}
MANIFEST_PATH = EVIDENCE_DEST / 'stage8_export_manifest.json'
MANIFEST_TEXT = json.dumps(EXPORT_MANIFEST, indent=2, sort_keys=True) + '\n'
MANIFEST_PATH.write_text(MANIFEST_TEXT, encoding='utf-8')
print(MANIFEST_TEXT, end='')
drive.flush_and_unmount()

# Second check: remount and re-hash what Drive actually stored after the flush.
drive.mount('/content/drive', force_remount=True)
expected = {entry['path']: entry['sha256'] for entry in EXPORT_MANIFEST['files']}
expected[MANIFEST_PATH.relative_to(DRIVE).as_posix()] = hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest()
stored = {p.relative_to(DRIVE).as_posix() for dest in (CHECKPOINT_DEST, EVIDENCE_DEST)
          for p in dest.rglob('*') if p.is_file()}
missing, extra = sorted(set(expected) - stored), sorted(stored - set(expected))
assert not missing and not extra, f'missing on Drive: {missing}; extra on Drive: {extra}'
mismatched = [path for path, digest in sorted(expected.items()) if sha256(DRIVE / path) != digest]
assert not mismatched, f'SHA-256 mismatch on Drive after remount: {mismatched}'
print(f'Drive verified after remount: {len(expected)} files match the manifest.')
print('Reminder: checkpoints are never committed to Git; only reviewed reports are.')